# 04 · TorchDistributor — M×N GPU

`local_mode=False`로 Spark worker 노드에 child 프로세스를 분산 spawn합니다. `num_processes=M*N`을 지정합니다. driver는 코디네이션만 담당하며 학습에 참여하지 않으므로, driver-side `log_system_metrics`만 켜면 idle metric만 잡힙니다. 따라서 `train_fn`은 rank-0 worker에서 `mlflow.start_run(run_id=..., log_system_metrics=True)`로 attach해 worker GPU 메트릭을 함께 기록합니다.

> 1×1은 [`02-launch_torch_distributor_1x1.ipynb`](02-launch_torch_distributor_1x1.ipynb), 1×N은 [`03-launch_torch_distributor_1xN.ipynb`](03-launch_torch_distributor_1xN.ipynb)에서 다룹니다.

**클러스터**: Multi-node Classic GPU, DBR 17.3 LTS ML. Driver `g5.12xlarge` + Workers `g5.12xlarge` × M. **Autoscaling OFF 필수**, Single node 토글 OFF. Serverless GPU는 multi-node 미지원입니다.

In [ ]:
%run ./00-setup

## 학습 함수 import

In [ ]:
import os
import sys

NOTEBOOK_PATH = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)
print(f"SCRIPT_DIR={SCRIPT_DIR}")


def td_train_fn(**kwargs):
    """TorchDistributor child에 보낼 thin wrapper.

    노트북 셀에서 module-level `train_fn`을 import해 TD에 직접 넘기면 cloudpickle이
    module reference로 직렬화하고, child 프로세스의 fresh sys.path에는 SCRIPT_DIR이
    없어 `from torch_distributor_trainer import train_fn`이 실패한다. inline 함수는
    bytecode가 by-value로 pickling되므로, child에서 sys.path를 보강한 뒤 lazy import해
    위 문제를 회피한다."""
    import sys

    sd = kwargs.get("script_dir")
    if sd and sd not in sys.path:
        sys.path.insert(0, sd)
    from torch_distributor_trainer import train_fn

    return train_fn(**kwargs)

## M×N GPU

In [ ]:
import mlflow
from pyspark.ml.torch.distributor import TorchDistributor

NUM_NODES = 2  # M (worker 노드 수)
NUM_GPUS_PER_NODE = 4  # N

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="td-MxN", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=NUM_NODES * NUM_GPUS_PER_NODE,
    local_mode=False,
    use_gpu=True,
).run(
    td_train_fn,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_path=os.path.join(CKPT_DIR, "td-MxN.pt"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    topology="MxN",
    script_dir=SCRIPT_DIR,
)